# Wine Quality - Grid Search (Binário)

Notebook de documentação técnica do tuning usado em `src/train.py` para regressão + threshold binário.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from xgboost import XGBRegressor

In [ ]:
df = pd.read_parquet('../data/processed/wine_processed.parquet')
X = df.drop(columns=['quality_raw','quality_binary'])
y_raw = df['quality_raw'].astype(float).values
y_bin = df['quality_binary'].astype(int).values

X_temp, X_test, y_raw_temp, y_raw_test, y_bin_temp, y_bin_test = train_test_split(
    X, y_raw, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)
X_train, X_val, y_raw_train, y_raw_val, y_bin_train, y_bin_val = train_test_split(
    X_temp, y_raw_temp, y_bin_temp, test_size=0.25, random_state=42, stratify=y_bin_temp
)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

print('Train/Val/Test:', len(X_train), len(X_val), len(X_test))

In [ ]:
def to_binary(y_score, t=6.5):
    return (np.asarray(y_score) >= t).astype(int)

def f1_bin_weighted(y_true_bin, y_pred_score, t=6.5):
    return f1_score(y_true_bin, to_binary(y_pred_score, t), average='weighted')

def cv_scorer(estimator, X_fold, y_raw_fold):
    y_pred = estimator.predict(X_fold)
    y_true_bin = (np.asarray(y_raw_fold) >= 7).astype(int)
    return f1_bin_weighted(y_true_bin, y_pred, t=6.5)

In [ ]:
models = {
    'xgboost_regressor': (
        XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
        {
            'regressor__n_estimators': [200, 400],
            'regressor__max_depth': [4, 6],
            'regressor__learning_rate': [0.03, 0.08],
            'regressor__subsample': [0.9],
            'regressor__colsample_bytree': [0.9],
        },
    ),
    'random_forest_regressor': (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            'regressor__n_estimators': [300, 500],
            'regressor__max_depth': [None, 20],
            'regressor__min_samples_leaf': [1, 3],
        },
    ),
    'hist_gradient_boosting_regressor': (
        HistGradientBoostingRegressor(random_state=42),
        {
            'regressor__max_iter': [250, 400],
            'regressor__learning_rate': [0.03, 0.08],
            'regressor__max_depth': [None, 10],
            'regressor__min_samples_leaf': [20, 40],
        },
    ),
}

bins = pd.qcut(y_raw_train, q=5, labels=False, duplicates='drop')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, (reg, grid) in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', reg)])
    gs = GridSearchCV(pipe, grid, scoring=cv_scorer, cv=cv.split(X_train, bins), n_jobs=-1)
    gs.fit(X_train, y_raw_train)

    y_val_score = gs.best_estimator_.predict(X_val)
    y_test_score = gs.best_estimator_.predict(X_test)

    results[name] = {
        'cv_binary_f1_weighted_default_t': float(gs.best_score_),
        'val_binary_f1_weighted_default_t': float(f1_bin_weighted(y_bin_val, y_val_score, t=6.5)),
        'test_binary_f1_weighted_default_t': float(f1_bin_weighted(y_bin_test, y_test_score, t=6.5)),
        'best_params': gs.best_params_,
    }

pd.DataFrame(results).T.sort_values('test_binary_f1_weighted_default_t', ascending=False)